In [36]:
import warnings
warnings.filterwarnings("ignore")

In [37]:
import sys
sys.path.append(r'C:\Users\julia\OneDrive\Escritorio\Trabajo\building_ml_models_for_protein_science\src')


In [38]:
from building_models.commons_functions.parsers_commons import ParsersCommons
from building_models.utils.constants import COLUMNS_TO_WORK
from building_models.utils.utils_functions import UtilsFunctions
import pandas as pd

In [39]:
path_export = "../../processed_dataset/"
path_input = "../../raw_dataset/"
metadata_file = "../../raw_dataset/raw_data_description.xlsx"
name_task = "antioxidant_classification"
name_source = "Ahmad et al"

- Read doc and labels

In [40]:
#independent datset from 1 to 392 non-antioxidant, and 393 to 465 antioxidant proteins  
df_data_ind = ParsersCommons.read_fasta_doc(f"{path_input}/{name_source}/independent dataset.txt")
df_data_ind['label'] = 0  # todos en 0 por defecto
df_data_ind.loc[392:464, 'label'] = 1  # filas 393 a 465 (índice 0-based)
df_data_ind.head()

,id,sequence,label
0,,MRSPSLAVAATTVLGLFSSSALAYYGNTTTVALTTTEFVTTCPYPT...,0
1,,MRLRRLALFPGVALLLAAARLAAASDVLELTDDNFESRISDTGSAG...,0
2,,MAPPQRHPQRSEQVLLLTLLGTLWGAAAAQIRYSIPEELEKGSFVG...,0
3,,MSELSDIRREYTLGELHSEDVPNDPMDLFNAWLEVVRDSQIQDPTA...,0
4,,MAGNAAVGVLALQGDVSEHISAFESAIQNLGLNIPVVPVRKAEQIL...,0


In [41]:
#from 1 to 100 non-antioxidant, and 101 to 200 antioxidant proteins  
df_data_train = ParsersCommons.read_fasta_doc(f"{path_input}/{name_source}/Training_dataset.txt")
df_data_train['label'] = 0  # todos en 0 por defecto
df_data_train.loc[100:199, 'label'] = 1  # filas 101 a 200 (índice 0-based)
df_data_train.head()

,id,sequence,label
0,,MQHGIKRVKLSEEAKRLKLEKDQIKIKNYRQLTDEIFELRANENYS...,0
1,,MARDIAAPPVPTNHQELISWVNEIAELTQPDAVVWCDGSEAEYERL...,0
2,,MPPRAPPAPGPRPPPRAAGRHGLSPLAPRPWRWLLLLALPAVCSAL...,0
3,,MSSEKEVQEKIATLQILQEEAEALQRRLMELEILENEYRKTLETLE...,0
4,,MTIKKIAVLTSGGDSQGMNAAVRAVVRSGLFYGLEVYGIQRGYQGL...,0


In [42]:
df_data = pd.concat([df_data_ind, df_data_train], ignore_index=True)
df_data = df_data.drop(columns=["id"])
df_data

,sequence,label
0,MRSPSLAVAATTVLGLFSSSALAYYGNTTTVALTTTEFVTTCPYPT...,0
1,MRLRRLALFPGVALLLAAARLAAASDVLELTDDNFESRISDTGSAG...,0
2,MAPPQRHPQRSEQVLLLTLLGTLWGAAAAQIRYSIPEELEKGSFVG...,0
3,MSELSDIRREYTLGELHSEDVPNDPMDLFNAWLEVVRDSQIQDPTA...,0
4,MAGNAAVGVLALQGDVSEHISAFESAIQNLGLNIPVVPVRKAEQIL...,0
...,...,...
660,MVAEVQKQAPPFKKTAVVDGIFEEISLEKYKGKYVVLAFVPLAFSF...,1
661,MTLVTQKAPNFIAPAILKNGKIMNNFDLKKYSNGQITVLFFWPMDF...,1
662,MVLVTYPAPDFTASAISCNGDIINNFNFKEFTNNQTSILFFWPMDF...,1
663,MTNFPKIGKTPPNFLTIGVYKKRLGKIRLSDYRGKKYVILFFYPAN...,1


- Checking duplicates

In [43]:
df_consistent_duplicates, df_errors, df_unique = ParsersCommons.processing_duplicated(
    df_data, group_seq= "sequence",
    label_col= "label")
df_consistent_duplicates.shape, df_errors.shape, df_unique.shape
#no duplicates

((0, 0), (0, 0), (665, 2))

- Checking labels

In [44]:
df_data["label"].value_counts()

label
0    492
1    173
Name: count, dtype: int64

- Working with metadata


In [45]:
metadata_file = ParsersCommons.read_metadata(metadata_file, name_source=name_source, columns_to_select=COLUMNS_TO_WORK)
metadata_file.head()

,name dataset,name source,type source,static-dynamic,license,reports constant updates,year of publication,last update date,download date,file format,protein format,category dataset,task,obtaining negative dataset,obtaining positive dataset,repository or server,publication,unit of measurement
0,Training_dataset.txt,Ahmad et al,Dataset,Static,No information,No,2022,2022-06-02,2026-04-07,txt,Sequence,Enzyme/protein classification,Antioxidant,Obtained from other databases,"Sampling from Swiss-Prot, Sampling from UniProt",https://github.com/salman-khan-mrd/Antioxident...,https://www.sciencedirect.com/science/article/...,No information
1,Indpndet_dataset.txt,Ahmad et al,Dataset,Static,No information,No,2022,2022-06-02,2026-04-07,txt,Sequence,Enzyme/protein classification,Antioxidant,Obtained from other databases,"Sampling from Swiss-Prot, Sampling from UniProt",https://github.com/salman-khan-mrd/Antioxident...,https://www.sciencedirect.com/science/article/...,No information


In [46]:
dict_metadata = ParsersCommons.create_metadata_from_file(metadata_file)
dict_metadata

{'name dataset': 'Training_dataset.txt;Indpndet_dataset.txt',
 'name source': 'Ahmad et al',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'reports constant updates': 'No',
 'year of publication': 2022,
 'last update date': Timestamp('2022-06-02 00:00:00'),
 'download date': Timestamp('2026-04-07 00:00:00'),
 'file format': 'txt',
 'protein format': 'Sequence',
 'category dataset': 'Enzyme/protein classification',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'Obtained from other databases',
 'obtaining positive dataset': 'Sampling from Swiss-Prot, Sampling from UniProt',
 'repository or server': 'https://github.com/salman-khan-mrd/Antioxident_proteins/tree/master/Datasets',
 'publication': 'https://www.sciencedirect.com/science/article/pii/S0208521620301200?via%3Dihub#bib0055',
 'unit of measurement': 'No information',
 'number_of_sources': 2,
 'processing_date': '2026-04-08 13:38:27'}

In [47]:
df_data["label"] = df_data["label"].astype(int)

In [48]:
dict_metadata['number_of_records'] = df_data.shape[0]
dict_metadata['number_of_collected_sequences'] = df_data.shape[0]
dict_metadata['number_of_unique_sequences'] = df_data.shape[0]
dict_metadata['positive_examples'] = df_data[df_data["label"] == 1].shape[0]
dict_metadata['negative_examples'] = df_data[df_data["label"] == 0].shape[0]
dict_metadata

{'name dataset': 'Training_dataset.txt;Indpndet_dataset.txt',
 'name source': 'Ahmad et al',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'reports constant updates': 'No',
 'year of publication': 2022,
 'last update date': Timestamp('2022-06-02 00:00:00'),
 'download date': Timestamp('2026-04-07 00:00:00'),
 'file format': 'txt',
 'protein format': 'Sequence',
 'category dataset': 'Enzyme/protein classification',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'Obtained from other databases',
 'obtaining positive dataset': 'Sampling from Swiss-Prot, Sampling from UniProt',
 'repository or server': 'https://github.com/salman-khan-mrd/Antioxident_proteins/tree/master/Datasets',
 'publication': 'https://www.sciencedirect.com/science/article/pii/S0208521620301200?via%3Dihub#bib0055',
 'unit of measurement': 'No information',
 'number_of_sources': 2,
 'processing_date': '2026-04-08 13:38:27',
 'number_of_records': 665,
 'number_of_collected_s

- Export data

In [49]:
UtilsFunctions.make_directory(f"{path_export}{name_task}/{name_source}")

In [50]:
UtilsFunctions.export_json(f"{path_export}{name_task}/{name_source}/metadata_{name_source}.json", dict_metadata)

In [51]:
df_data.to_csv(f"{path_export}{name_task}/{name_source}/processed_data.csv", index=False)